## Results

In [84]:
%run ../utils/sampling.py
import pandas as pd

product_info_df = pd.read_csv('../data/cleaned/product-features.csv')
departments = pd.read_csv('../dataset/departments.csv')
products = pd.read_csv('../dataset/products.csv')

full_products_df = product_info_df.merge(products, on='product_id', how='left')

product_names_to_search = [
    "Coke",
    "Bananas",
    "Organic Whole Milk",
    "Plain Bagels",
    "Spaghetti"
]

top_ids = get_top_product_ids(full_products_df, product_names_to_search)
print(top_ids)

found_products = search_products(full_products_df, "Coke")
print(found_products.head(1))

found_products = search_products(full_products_df, "Eggo")
print(found_products.head(1))

found_products = search_products(full_products_df, "potato chips")
print(found_products.head(1))

{'Coke': {'query': 'Coke', 'matched_name': 'Diet Coke', 'product_id': 43631, 'order_penetration_pct': 0.2090284098225933}, 'Bananas': {'query': 'Bananas', 'matched_name': 'Banana', 'product_id': 24852, 'order_penetration_pct': 14.69933191782944}, 'Organic Whole Milk': {'query': 'Organic Whole Milk', 'matched_name': 'Organic Whole Milk', 'product_id': 27845, 'order_penetration_pct': 4.289592686991776}, 'Plain Bagels': {'query': 'Plain Bagels', 'matched_name': 'Plain Bagels', 'product_id': 20738, 'order_penetration_pct': 0.3195770658507922}, 'Spaghetti': {'query': 'Spaghetti', 'matched_name': 'Spaghetti', 'product_id': 32734, 'order_penetration_pct': 0.49186997686379}}
   product_name  product_id  order_penetration_pct
0  Coke Classic       16696               0.335068
             product_name  product_id  order_penetration_pct
0  Eggo Homestyle Waffles       30696               0.276216
                      product_name  product_id  order_penetration_pct
0  Sea Salt & Vinegar Potato C

In [ ]:
product_ids = [
    info["product_id"] 
    for info in top_ids.values() 
    if info is not None
]

product_ids.append(16696)
product_ids.append(30696)
product_ids.append(40709)
print(product_ids)
sampled_products_df = pd.DataFrame(product_ids, columns=['product_id'])
sampled_products_df.to_csv("../results/sampled-products.csv", index=None)

[43631, 24852, 27845, 20738, 32734, 16696, 30696]


In [3]:
%run ../utils/pairwise.py

# Get product_ids as a series
products = product_ids.copy()
product_df = pd.read_csv('../data/cleaned/product-info-full.csv')
orders_full_df = pd.read_csv('../dataset/order_products__prior.csv')

order_product_df = orders_full_df[['order_id', 'product_id']]

# Compute pairwise probabilties for focus products
compute_pairwise_probabilities_sample(order_product_df, 
    products,
    output_csv="../results/products-pairwise.csv",
    batch_size=1000
)

100%|██████████| 1/1 [00:04<00:00,  4.20s/it]

Completed computation. Saved to ../results/products-pairwise.csv


In [ ]:
%run ../utils/substitutes.py

dept1_df = pd.read_csv('../data/cleaned/pairwise-dept1.csv')
dept3_df = pd.read_csv('../data/cleaned/pairwise-dept3.csv')
dept4_df = pd.read_csv('../data/cleaned/pairwise-dept4.csv')
dept7_df = pd.read_csv('../data/cleaned/pairwise-dept7.csv')
dept9_df = pd.read_csv('../data/cleaned/pairwise-dept9.csv')
dept16_df = pd.read_csv('../data/cleaned/pairwise-dept16.csv')
dept19_df = pd.read_csv('../data/cleaned/pairwise-dept16.csv')

pairwise_df = pd.concat([dept1_df, dept3_df, dept4_df, dept7_df, dept9_df, dept16_df, dept19_df], ignore_index=True)

similarity_df = pd.read_csv('../data/cleaned/product-similiarity.csv')

compute_sub_score(
    product_df,
    pairwise_df,
    products,
    similarity_df,
    "../results/raw-substitutes.csv")

Completed substitute calculations. Saved to ../results/raw-substitutes.csv


In [82]:
%run ../utils/substitutes.py

substitutes_df = pd.read_csv("../results/raw-substitutes.csv")

# Find the best substitution score threshold which will identify a product as a true substitute
best_threshold = find_best_threshold(substitutes_df)

# Add a new column 'identified_substitute' based on the threshold
substitutes_df['identified_substitute'] = substitutes_df['score'] >= best_threshold

# Compute transferability %
results = compute_transferability(substitutes_df, top_n=5)
results.to_csv("../results/substitutes-transfer.csv", index=False)

 Best threshold determined as: 0.398 and adjusted to: 0.41074705927046157



In [ ]:
%run ../utils/results.py

# Print the results
show_sub_results(results)


Product: Coke Classic  (ID: 16696)


,sub_name,transferability_pct,score,aisle
0,Classic Soda,0.149826,0.701174,soft drinks
1,Coke Zero,0.144124,0.674490,soft drinks
2,Coke,0.141851,0.663852,soft drinks
3,Cherry Coke,0.134633,0.630075,soft drinks
4,Vanilla Coke Zero,0.130740,0.611854,soft drinks



Product: Plain Bagels  (ID: 20738)


,sub_name,transferability_pct,score,aisle
0,Plain Mini Bagels,0.154851,0.737898,breakfast bakery
1,Plain Pre-Sliced Bagels,0.154590,0.736655,breakfast bakery
2,Assorted Bagels,0.145627,0.693944,breakfast bakery
3,Bagels Plain Presliced,0.143973,0.686062,breakfast bakery
4,Onion Bagels,0.138857,0.661686,breakfast bakery



Product: Banana  (ID: 24852)


...

,sub_name,transferability_pct,score,aisle
0,Homestyle Waffles,0.145661,0.699007,frozen breakfast
1,Homestyle Belgian Waffles,0.141890,0.680906,frozen breakfast
2,Eggo Buttermilk Waffles,0.138502,0.664652,frozen breakfast
3,Eggo Thick & Fluffy Original Waffles,0.138212,0.663260,frozen breakfast
4,Buttermilk Waffles,0.134741,0.646601,frozen breakfast



Product: Spaghetti  (ID: 32734)


,sub_name,transferability_pct,score,aisle
0,Spaghetti Pasta,0.193824,0.789977,dry pasta
1,Thin Spaghetti Pasta,0.172642,0.703646,dry pasta
2,Whole Grain Spaghetti,0.161338,0.657572,dry pasta
3,Whole Wheat Spaghetti,0.155815,0.635062,dry pasta
4,Penne Rigate,0.106358,0.433486,dry pasta



Product: Diet Coke  (ID: 43631)


,sub_name,transferability_pct,score,aisle
0,Diet Cola,0.184405,0.680563,soft drinks
1,Diet Pepsi Soda,0.152335,0.562203,soft drinks
2,Soda,0.118957,0.439019,soft drinks
3,Fridge Pack Cola,0.112507,0.415215,soft drinks
4,Ginger Ale,0.112360,0.414673,soft drinks


In [ ]:
%run ../utils/complements.py

sample_df = pd.read_csv("../results3/sampled-products.csv")
pairwise_df = pd.read_csv("../results3/products-pairwise.csv")

num_orders, min_pij = get_min_pij()

lift_df = compute_lift(focus_products, pairwise_df, min_pij=min_pij, total_orders=num_orders)
complements_df = compute_hybrid_score(lift_df, focus_products, top_n=10)
cii_df = compute_complement_impact_index(complements_df, pairwise_df)
network_df = compute_network_enhanced_impact(cii_df, pairwise_df)
network_df.to_csv("../results/complements-impact.csv", index=False)

show_comp_results(network_df)

Computing lift: 100%|██████████| 6/6 [00:01<00:00,  4.40it/s]


In [ ]:
product_info_df = pd.read_csv('../data/cleaned/product-features.csv')

totals, details = compute_total_impact(
    pairwise_impact_df=network_df,        # or original_df depending on which impact you want to aggregate
    penetration_df=product_info_df, # or series
    method="topk_weighted",
    top_k=5,
    normalize=True,
    return_details=True
)
totals.to_csv("../results/complements-total-impact.csv", index=False)


   product_id  total_impact  total_weight  n_complements  mean_pair_impact  \
0       16696      0.000546      0.000801              5          0.706172   
5       43631      0.000448      0.000883              5          0.574459   
1       20738      0.000316      0.000873              5          0.363194   
3       27845      0.000113      0.000585              5          0.190997   
2       24852      0.000101      0.000515              5          0.196059   
4       32734      0.000060      0.000878              5          0.073045   

   total_impact_norm  
0           1.000000  
5           0.820555  
1           0.578623  
3           0.206769  
2           0.184757  
4           0.109187  
